# Phụ lục B: Đối chiếu yêu cầu đồ án và giới hạn thực nghiệm

Bảng dưới đây đối chiếu các yêu cầu kỹ thuật của Milestone 1, Milestone 2, yêu cầu trình bày Jupyter Book và các giới hạn thực tế ghi nhận trên tập dữ liệu 25.000 bài báo:

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
sa = m1["source_audit"]
rows = [
    ("M1", "Khám phá đủ dải mã của 5 thành viên và tích hợp OAI-PMH 2025", "Đạt 25.000 / 25.000 bài (0 trùng chéo)", "Sử dụng oaipmh.arxiv.org và chạy bù --resume trên Colab CPU cho phân vùng TRI (24120470)"),
    ("M1", "Tải nguồn đa phiên bản, lọc ảnh và kiểm toán độc lập", f"{fmt_int(m1['source_verified'])} / {fmt_int(scope['versions'])} phiên bản xác minh", f"Giảm 89,30% dung lượng (byte); còn 11 phiên bản chờ từ TRIET và {fmt_int(sa.get('not_found_verified', 0))} bản HTTP 404"),
    ("M1", "Thu thập trích dẫn S2 theo lô/phân trang và xuất bảng chuẩn", f"{fmt_int(m1['verified_reference_papers'])} / {fmt_int(scope['papers'])} bài S2 xác minh", f"{fmt_int(m1['working_reference_rows'])} dòng working (0 mồ côi) và {fmt_int(m1['arxiv_reference_rows'])} cạnh có đích arXiv"),
    ("M1", "Đo đầy đủ 7 chỉ số hiệu năng trình thu thập trên Colab CPU", "Đạt đủ 7/7 chỉ số bắt buộc", "Trung bình 8,06s/bài (4 phiên đầy đủ) và 8,35s/bài (đủ 25.000 bài); Peak RSS 1.227,73 MiB"),
    ("M2", "Gom tệp đa nguồn và phân giải nhiều tệp gốc biên dịch", "Xác định thêm 1.706 bản đa tệp nhờ đối sánh tiêu đề", "26 phiên bản thiếu \\documentclass; 271 phiên bản chứa biến macro trong đường dẫn \\input"),
    ("M2", "Xây dựng cây phân cấp (sentence, formula, figure, list)", "97,34% phiên bản có TeX dựng được cây hợp lệ", "791 phiên bản (9.087 lượt cảnh báo) đặt \\item trong môi trường tự định nghĩa được giữ ở mức partial"),
    ("M2", "Chuẩn hóa LaTeX/toán học và khử trùng hai tầng", f"Elements -{fmt_pct(m2['deduplication']['elements']['reduction_rate'], 2, True)}; BibTeX -{fmt_pct(m2['deduplication']['bibtex']['reduction_rate'], 2, True)}", "Giảm 46,95% elements và 50,41% bibtex trên nhóm bài đa phiên bản; giữ .bbl và đồng bộ khóa \\cite{}"),
    ("M2", "Kiểm chuẩn 5 bảng ngoài bộ nhớ bằng chỉ mục SQLite", "structural_valid = True (0 lỗi)", f"Elements {fmt_int(m2['rows']['elements'])}; Hierarchy {fmt_int(m2['rows']['hierarchy'])}; BibTeX {fmt_int(m2['rows']['bibtex'])}"),
    ("M2", "Đo hiệu năng 5 phiên độc lập trên Google Colab CPU-only", "Đạt 5/5 phiên chạy mới (reused = 0)", f"Tổng {fmt_num(bench['totals']['total_profiler_sec'], 2)}s ({fmt_num(bench['totals']['avg_sec_per_unique_publication'], 4)}s/bài); Peak RSS {fmt_num(bench['totals']['max_session_peak_rss_mib'], 2)} MiB"),
    ("Book", "Báo cáo thực thi động bằng Tiếng Việt trên GitHub Pages", "Biên dịch --all -W đạt 0 cảnh báo", "Thực thi trực tiếp mã nguồn parser, bảng biểu chuẩn học thuật và biểu đồ trực quan ở cùng địa chỉ URL cố định"),
]
display(style_table(
    pd.DataFrame(rows, columns=["Giai đoạn", "Hạng mục yêu cầu", "Kết quả xác minh (25.000 bài)", "Ghi chú kỹ thuật và giới hạn thực nghiệm"]),
    caption="Bảng B.1. Đối chiếu yêu cầu đồ án và các giới hạn thực nghiệm trên 25.000 bài báo.",
))

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`README.md`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/README.md).
- Mã nguồn triển khai: [`src/parser/validation.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/validation.py).